In [2]:
import pandas as pd

df = pd.read_csv("data/non gene expression/12_CCLE_metabolomics_20190502.csv")

print("=== SHAPE ===")
print(df.shape)

print("\n=== COLUMNS & DTYPES ===")
print(df.dtypes.to_string())

print("\n=== HEAD (3) ===")
print(df.head(3).to_string())

print("\n=== NULL COUNTS ===")
null_counts = df.isnull().sum()
print(null_counts[null_counts > 0].to_string())
print(f"Total null cells: {null_counts.sum():,}")

print("\n=== UNIQUE VALUE COUNTS PER COLUMN ===")
for col in df.columns[:10]:
    print(f"  {col}: {df[col].nunique()} unique")

print("\n=== FORMAT GUESS ===")
print(f"Rows: {df.shape[0]:,}  Cols: {df.shape[1]:,}")
if df.shape[1] > df.shape[0]:
    print("→ Likely WIDE format (cell lines as columns or metabolites as columns)")
else:
    print("→ Likely LONG format (one row per measurement)")

print("\n=== CELL LINE ID CHECK ===")
# Check first column and index for ACH / CVCL / cell line name format
print(f"  First column name: {df.columns[0]}")
print(f"  First column sample values: {df.iloc[:5, 0].tolist()}")
print(f"  ACH- format present: {df.iloc[:, 0].astype(str).str.startswith('ACH').any()}")
print(f"  CVCL format present: {df.iloc[:, 0].astype(str).str.startswith('CVCL').any()}")

print("\n=== METABOLITE COLUMNS SAMPLE ===")
# Show first 10 non-ID column names to understand metabolite naming
print(df.columns[:20].tolist())

print("\n=== NUMERIC SUMMARY (first 5 metabolite cols) ===")
numeric_cols = df.select_dtypes(include='number').columns[:5]
print(df[numeric_cols].describe().to_string())

print("\n=== MISSINGNESS BY ROW (cell line coverage) ===")
row_null_pct = df.isnull().mean(axis=1)
print(f"  Mean % missing per cell line: {row_null_pct.mean()*100:.1f}%")
print(f"  Cell lines with >50% missing: {(row_null_pct > 0.5).sum()}")
print(f"  Cell lines with 0% missing:   {(row_null_pct == 0).sum()}")

print("\n=== MISSINGNESS BY COLUMN (metabolite coverage) ===")
col_null_pct = df.isnull().mean(axis=0)
numeric_null = col_null_pct[df.select_dtypes(include='number').columns]
print(f"  Mean % missing per metabolite: {numeric_null.mean()*100:.1f}%")
print(f"  Metabolites with >50% missing: {(numeric_null > 0.5).sum()}")
print(f"  Metabolites with 0% missing:   {(numeric_null == 0).sum()}")

=== SHAPE ===
(928, 227)

=== COLUMNS & DTYPES ===
CCLE_ID                                                            object
DepMap_ID                                                          object
2-aminoadipate                                                    float64
3-phosphoglycerate                                                float64
alpha-glycerophosphate                                            float64
4-pyridoxate                                                      float64
aconitate                                                         float64
adenine                                                           float64
adipate                                                           float64
alpha-ketoglutarate                                               float64
AMP                                                               float64
citrate                                                           float64
isocitrate                                                   

In [4]:
import pandas as pd

df = pd.read_csv("data/non gene expression/12_CCLE_metabolomics_20190502.csv")

# ── 1. Identify the one null DepMap_ID row ──────────────────────────────────
print("=== NULL DepMap_ID ROW ===")
print(df[df["DepMap_ID"].isna()][["CCLE_ID", "DepMap_ID"]])

# ── 2. Confirm log2 scale ───────────────────────────────────────────────────
import numpy as np
metabolite_cols = df.columns[2:]

print("\n=== VALUE RANGE CONFIRMATION ===")
vals = df[metabolite_cols].values.flatten()
print(f"  Global min:    {vals.min():.3f}")
print(f"  Global max:    {vals.max():.3f}")
print(f"  Global mean:   {vals.mean():.3f}")
print(f"  Global median: {np.median(vals):.3f}")
print(f"  Global std:    {vals.std():.3f}")
# If truly log2, back-transforming mean ~5.9 gives ~60 ion counts — plausible
print(f"  Back-transform check: 2^mean = {2**vals.mean():.1f} (should be plausible ion count)")

# ── 3. Overlap with RNA cell lines ──────────────────────────────────────────
print("\n=== CELL LINE OVERLAP WITH RNA ===")
profiles = pd.read_csv("data/nomenclature/8_DepMap_OmicsProfiles.csv")
rna_ach = set(profiles[profiles["Datatype"] == "rna"]["ModelID"].dropna())
metab_ach = set(df["DepMap_ID"].dropna())

overlap = metab_ach & rna_ach
only_metab = metab_ach - rna_ach
only_rna = rna_ach - metab_ach

print(f"  Metabolomics cell lines:          {len(metab_ach):,}")
print(f"  RNA cell lines:                   {len(rna_ach):,}")
print(f"  Both RNA + metabolomics:          {len(overlap):,}")
print(f"  Metabolomics only (no RNA):       {len(only_metab):,}")
print(f"  RNA only (no metabolomics):       {len(only_rna):,}")

# ── 4. Metabolite category breakdown ───────────────────────────────────────
print("\n=== METABOLITE CATEGORIES ===")
# Rough categorisation by naming convention
lipid_tags = ["LPC", "LPE", "PC", "SM", "DAG", "CE", "TAG"]
amino_acids = ["glycine","alanine","serine","threonine","methionine","aspartate",
               "glutamate","asparagine","glutamine","histidine","arginine","lysine",
               "valine","leucine","isoleucine","phenylalanine","tyrosine","tryptophan",
               "proline","ornithine","citrulline","cystathionine"]
carnitines  = [c for c in metabolite_cols if "carnitine" in c.lower()]
nucleotides = ["AMP","CMP","GMP","UMP","UMP","dCMP","cAMP","NAD","NADP"]

lipid_cols  = [c for c in metabolite_cols if any(t in c for t in lipid_tags)]
aa_cols     = [c for c in metabolite_cols if c.lower() in amino_acids]

print(f"  Total metabolites:   {len(metabolite_cols)}")
print(f"  Lipid species:       {len(lipid_cols)}")
print(f"  Amino acids:         {len(aa_cols)}")
print(f"  Acylcarnitines:      {len(carnitines)}")
print(f"  Nucleotides (rough): {sum(1 for c in metabolite_cols if c in nucleotides)}")
print(f"  Other/unclassified:  {len(metabolite_cols) - len(lipid_cols) - len(aa_cols) - len(carnitines)}")

# ── 5. Variance across cell lines per metabolite ────────────────────────────
print("\n=== TOP 10 MOST VARIABLE METABOLITES ===")
metabolite_std = df[metabolite_cols].std().sort_values(ascending=False)
print(metabolite_std.head(10).to_string())

print("\n=== TOP 10 LEAST VARIABLE METABOLITES ===")
print(metabolite_std.tail(10).to_string())

=== NULL DepMap_ID ROW ===
         CCLE_ID DepMap_ID
553  OC315_OVARY       NaN

=== VALUE RANGE CONFIRMATION ===
  Global min:    2.976
  Global max:    9.099
  Global mean:   5.877
  Global median: 5.884
  Global std:    0.402
  Back-transform check: 2^mean = 58.8 (should be plausible ion count)

=== CELL LINE OVERLAP WITH RNA ===
  Metabolomics cell lines:          927
  RNA cell lines:                   1,479
  Both RNA + metabolomics:          911
  Metabolomics only (no RNA):       16
  RNA only (no metabolomics):       568

=== METABOLITE CATEGORIES ===
  Total metabolites:   225
  Lipid species:       89
  Amino acids:         22
  Acylcarnitines:      14
  Nucleotides (rough): 8
  Other/unclassified:  100

=== TOP 10 MOST VARIABLE METABOLITES ===
1-methylnicotinamide                        1.346944
acetylcholine                               0.913129
taurodeoxycholate/taurochenodeoxycholate    0.857743
phosphocreatine                             0.777652
2-deoxycytidine      

In [6]:
import pandas as pd

df = pd.read_csv("data/non gene expression/12_CCLE_metabolomics_20190502.csv")
metabolite_cols = df.columns[2:]

# How many metabolites survive a std > 0.4 cutoff?
metabolite_std = df[metabolite_cols].std()
high_var = metabolite_std[metabolite_std > 0.4]
low_var  = metabolite_std[metabolite_std <= 0.4]

print(f"Metabolites with std > 0.4:  {len(high_var)}  ({len(high_var)/len(metabolite_cols)*100:.1f}%)")
print(f"Metabolites with std <= 0.4: {len(low_var)}  ({len(low_var)/len(metabolite_cols)*100:.1f}%)")
print(f"\nHigh-variance metabolites:\n{high_var.sort_values(ascending=False).to_string()}")

# Also check CCLE_ID format — does it match sample_info CCLE_Name for joining?
sample_info = pd.read_csv("data/nomenclature/9_DepMap_sample_info.csv")
metab_ccle  = set(df["CCLE_ID"].dropna())
info_ccle   = set(sample_info["CCLE_Name"].dropna())

match  = metab_ccle & info_ccle
no_match = metab_ccle - info_ccle
print(f"\n=== CCLE_ID vs sample_info CCLE_Name ===")
print(f"  Matching:     {len(match):,}")
print(f"  No match:     {len(no_match):,}")
if no_match:
    print(f"  Examples with no match: {list(no_match)[:5]}")

Metabolites with std > 0.4:  73  (32.4%)
Metabolites with std <= 0.4: 152  (67.6%)

High-variance metabolites:
1-methylnicotinamide                                              1.346944
acetylcholine                                                     0.913129
taurodeoxycholate/taurochenodeoxycholate                          0.857743
phosphocreatine                                                   0.777652
2-deoxycytidine                                                   0.717937
palmitoylcarnitine                                                0.642934
hypoxanthine                                                      0.639263
stearoylcarnitine                                                 0.593008
oleylcarnitine                                                    0.592222
N-carbamoyl-beta-alanine                                          0.591903
C58:7 TAG                                                         0.580400
adenosine                                                       